In [ ]:
# Run this cell to install ChromaDB if desired
try:
    assert version('chromadb') == '0.4.17'
except:
    !pip install chromadb==0.4.17
try:
    assert version('pysqlite3') == '0.5.2'
except:
    !pip install pysqlite3-binary==0.5.2
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
import chromadb

In [ ]:
# Load the dataset
import pandas as pd
reviews = pd.read_csv("womens_clothing_e-commerce_reviews.csv")

# Display the first few entries
reviews.head()

In [ ]:
# Start coding here
# Use as many cells as you need.
from openai import OpenAI
import numpy as np

client = OpenAI()

# Clean data: drop rows with no review text
reviews_clean = reviews.dropna(subset=['Review Text']).reset_index(drop=True)
texts = reviews_clean['Review Text'].tolist()

# Generate embeddings
response = client.embeddings.create(input=texts, model="text-embedding-3-small")
embeddings = [data.embedding for data in response.data]

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce to 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.5, s=2)
plt.title("2D Visualization of Clothing Reviews")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
import chromadb

# Initialize ChromaDB
chroma_client = chromadb.Client()

# Use get_or_create_collection to avoid the UniqueConstraintError
collection = chroma_client.get_or_create_collection(name="clothing_reviews")

# Check if the collection is empty before adding to avoid duplicate IDs
if collection.count() == 0:
    collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=[str(i) for i in range(len(texts))]
    )

def get_most_similar(query_text, n=3):
    # 1. Embed the query text using the SAME OpenAI model
    query_response = client.embeddings.create(
        input=[query_text], 
        model="text-embedding-3-small"
    )
    query_embedding = query_response.data[0].embedding
    
    # 2. Query the collection using 'query_embeddings'
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n
    )
    return results['documents'][0]

# Now run the search
first_review = "Absolutely wonderful - silky and sexy and comfortable"
most_similar_reviews = get_most_similar(first_review, n=3)

print(most_similar_reviews)